<div style="text-align: justify; max-width: 90ch">

# Lecture 01e: Run a model with `transformers`, what Ollama hides

**Status: Optional / career-track.**

Every notebook so far talked to Ollama, a server that loads a model and answers over HTTP. This notebook removes the server:
it loads the same family of model directly into Python with Hugging Face's [`transformers`](https://huggingface.co/docs/transformers/index) library,
the way researchers and model builders do, and shows the pieces Ollama handles for you.

**What this notebook covers**

- What `transformers` is, and the problem it solves.
- Loading a model's weights and tokenizer from the Hub into a Python process.
- The **chat template**: how roles become one string of tokens.
- Generating tokens one by one, and what that costs on a CPU.
- What an inference engine (Ollama, llama.cpp) adds on top.

Needs the optional group `uv sync --group transformers-run` (PyTorch, about 200 MB to download) and a one-time download of `Qwen/Qwen3-0.6B` (about 1.5 GB).

</div>

In [1]:
import os
import time
from pathlib import Path

import ollama
import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()  # keep the weight-loading bar out of saved outputs

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. What `transformers` is, and what it solves

A trained model is published as files:

- a **config** that describes the network (how many layers, how wide)
- the **weights** (the numbers learned in training)
- the tokenizer (`lec_01a`)

Files are not a program. To get an answer, code must:

- build the network the config describes;
- load the weights into it;
- turn text into token ids and back;
- run the generation loop.

Every model family (Qwen, Llama, Gemma, Mistral) has its own architecture, so without a shared library each would need its own code,
often the research code released with the paper. `transformers`, whose tokenizer you used in `lec_01a`, is that shared code:
hundreds of architectures behind one interface, reading models straight from the Hugging Face Hub (`lec_01b`). Its code is open source (Apache 2.0).

</div>

<div style="text-align: justify; max-width: 90ch">

Its main names, and the closest thing you already know:

| `transformers` name | What it does | Closest thing you know |
| --- | --- | --- |
| `AutoTokenizer` | text to token ids and back, for one model | the tokenizer from `lec_01a` |
| `AutoModelForCausalLM` | builds the network and loads its weights | a fitted scikit-learn model, loaded instead of trained |
| `from_pretrained` | downloads a model once, then loads it from a cache | `pd.read_csv`, for a model |
| `generate` | runs the next-token loop | Ollama's `client.chat`, one level down |

**Auto** means the right model code is picked from the model's config, so the same two lines load any supported model. A **causal language model** (causal LM) predicts the next token from the ones before it:
every chat model in this course is one.

</div>

<div style="text-align: justify; max-width: 90ch">

`transformers` does its arithmetic with [PyTorch](https://pytorch.org/), a library for **tensors**: arrays like NumPy's that can also live on a GPU.
That is why this notebook needs the optional group. What you get, and what you pay:

- **New models first.** A new model's Hub page usually shows `transformers` code; engines such as llama.cpp convert those files to GGUF (`lec_01b`) afterwards.
- **The model itself.** You can look inside the network, control every step of generation, and fine-tune the model, training it further on your own data (`lec_01b`);
  Ollama only runs it.
- **The cost:**
  - a large dependency
  - full-precision weights that need more memory
  - a slower generation loop; section 5 times it against Ollama

Use `transformers` to study or change a model; use a server such as Ollama to build a system on one.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Load the weights

`from_pretrained` downloads the model's folder from the Hub once, into the Hugging Face cache (`~/.cache/huggingface/hub` by default),
and builds the network in memory. The weights come as **safetensors** files, a format that holds only numbers: older PyTorch files could run code when loaded, a risk with files downloaded from the internet.

`Qwen3-0.6B` is the smallest of the family; it is the model behind `qwen3:0.6b`, before quantization.

</div>

In [3]:
HF_MODEL = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL, dtype="auto"
)  # bf16 weights as published
model.eval()

n_params = sum(p.numel() for p in model.parameters())
bytes_per_param = next(model.parameters()).element_size()

print(
    f"{n_params / 1e6:.0f} M parameters x {bytes_per_param} bytes = {n_params * bytes_per_param / 1e9:.2f} GB in memory"
)
print("device:", next(model.parameters()).device)

596 M parameters x 2 bytes = 1.19 GB in memory
device: cpu


<div style="text-align: justify; max-width: 90ch">

The arithmetic from `lec_01b` section 4, live: parameters times bytes. `dtype="auto"` keeps the precision the model was published in,
here **bfloat16**, a 2-byte number format made for neural networks; Ollama's `Q4_K_M` build of the same model uses about 0.6 bytes per weight instead of 2.

`model.eval()` switches off behaviour used only in training, and `device: cpu` says the weights sit in main memory: this notebook does not ask for a GPU.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. The chat template

A model reads one sequence of tokens; there are no "roles" inside it. The **chat template** is a recipe, shipped with the tokenizer, that lays the messages out with **special tokens**:
tokens reserved for structure, which the model was trained to recognise. Ollama applies it silently for every call; here you can look at it.

`apply_chat_template` takes the same list of messages as `client.chat` in `lec_01c`. `tokenize=False` returns the text, so you can read it, instead of the token ids a model needs.

</div>

In [4]:
messages = [
    {
        "role": "system",
        "content": "You are a teaching assistant for a Python course. Answer in one sentence.",
    },
    {"role": "user", "content": "What is a DataFrame?"},
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # append the tokens that mean "assistant, your turn"
    enable_thinking=False,  # the template-level version of think=False
)

print(prompt_text)

<|im_start|>system
You are a teaching assistant for a Python course. Answer in one sentence.<|im_end|>
<|im_start|>user
What is a DataFrame?<|im_end|>
<|im_start|>assistant
<think>

</think>




<div style="text-align: justify; max-width: 90ch">

`<|im_start|>` and `<|im_end|>` are single tokens that mark the turns; the empty `<think>` block is how this template switches reasoning off.
That is what `think=False` did in every earlier notebook, and why a Hub file with a different template (`lec_01b` section 6b) can behave differently.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Generate

`generate` runs the loop from `lec_01a` section 2: predict the next token, append it, repeat. Its arguments are settings you know under other names:

- `max_new_tokens=60`: Ollama's `num_predict`.
- `do_sample=False`: always take the most likely token, which is temperature 0.
- `return_tensors="pt"`: the tokenizer returns the token ids as a PyTorch tensor, the input `generate` expects.

`torch.no_grad()` tells PyTorch to skip the bookkeeping it keeps only for training, which saves memory. The loop runs in Python on the CPU, without an inference engine's optimisations.

</div>

In [5]:
inputs = tokenizer(prompt_text, return_tensors="pt")

print("prompt tokens:", inputs["input_ids"].shape[1])

started = time.time()

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=60, do_sample=False)

elapsed = time.time() - started

new_tokens = output_ids[0, inputs["input_ids"].shape[1] :]

print(tokenizer.decode(new_tokens, skip_special_tokens=True))
print(
    f"\n{len(new_tokens)} tokens in {elapsed:.1f} s -> {len(new_tokens) / elapsed:.1f} tokens/s (transformers, CPU)"
)

prompt tokens: 37


A DataFrame is a 2D data structure that represents a table of data in a dataset.

20 tokens in 0.9 s -> 21.2 tokens/s (transformers, CPU)


<div style="text-align: justify; max-width: 90ch">

## 5. The same model through Ollama

Same prompt, same weights, quantized to `Q4_K_M` (`lec_01b`), through the server: `qwen3:0.6b` is Ollama's build of `Qwen/Qwen3-0.6B`, called with `client.chat` as in `lec_01a`.

</div>

In [6]:
client = ollama.Client(host=OLLAMA_HOST)

check_ollama(OLLAMA_HOST, ["qwen3:0.6b"])

response = client.chat(
    model="qwen3:0.6b",
    messages=messages,
    think=False,
    options={"temperature": 0, "num_predict": 60},
)

print(response.message.content)
print(
    f"\n{response.eval_count} tokens -> {response.eval_count / (response.eval_duration / 1e9):.1f} tokens/s (Ollama)"
)

Ollama OK at http://localhost:11434; models ready: qwen3:0.6b


A DataFrame is a 2D data structure that represents a table, with rows and columns, allowing for efficient data manipulation and analysis.

28 tokens -> 224.4 tokens/s (Ollama)


<div style="text-align: justify; max-width: 90ch">

Compare the two tokens/s lines. In one example run, `transformers` produced about 21 tokens per second on the CPU and Ollama about 224,
on a machine where Ollama found a GPU. With the GPU switched off, Ollama was still several times faster in one example run:
quantization and a C++ engine, not only the hardware.

Compare the two answers too: they often begin with the same words, then part. Sampling is off in both, so a difference is not chance:
the quantized weights shift the probabilities slightly, and at some token the most likely choice can change.

</div>

<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Compare the two speeds and the two answers. What Ollama added, in order of how much it matters on a laptop:

| Ollama does | Instead of |
| --- | --- |
| runs a **quantized** build (4-bit) in a C++ engine (`llama.cpp`) | full-precision weights in Python |
| offloads to the **GPU** when one fits, transparently | you choosing devices and dtypes |
| keeps the model **loaded** between calls, serves several programs | one process, one model, reload every run |
| applies the **chat template**, handles thinking, JSON schemas, streaming | you writing that code |
| a stable **HTTP API** any language can call | Python only |

That list is also what a hosted API sells you, plus hardware you do not own.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- `transformers` is the shared code that turns a model's published files (config, weights, tokenizer) into a working network,
  for hundreds of architectures, on PyTorch.
- A model is weights plus a tokenizer plus a chat template; `transformers` loads all three into your process.
- The chat template turns roles into special tokens; `enable_thinking=False` is `think=False` at that level.
- Generation is a loop over tokens; on a CPU in Python it is slow, which is what inference engines fix.
- Ollama adds quantization, GPU offload, a resident server and a stable API. Use `transformers` when you need the model itself (research, fine-tuning);
  use a server when you need a system.

Back to the mandatory path: **`lec_01f`** (optional) or **lecture 2**.

</div>